# By-Client Strategy Comparison

Use this notebook to compare the latest available by-client run from each preprocessing strategy and rank the combined candidates from best to worst under a client-disjoint evaluation.

In [36]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
if (cwd / 'zscore').exists() and (cwd / 'client_zscore').exists():
    byclient_root = cwd
elif (cwd / 'experiments' / 'byclient').exists():
    byclient_root = cwd / 'experiments' / 'byclient'
else:
    raise FileNotFoundError('Could not locate the experiments/byclient folder from the current working directory.')

strategies = ['zscore', 'client_zscore', 'magnitude_features', 'magnitude_only', 'robust_clip']
ranking_columns = ['pr_auc', 'miss_rate', 'far', 'balanced_accuracy', 'f1']
ranking_ascending = [False, True, True, False, False]

strategy_runs = []
for strategy in strategies:
    results_root = byclient_root / strategy / 'results'
    run_dirs = sorted([path for path in results_root.glob('run_*') if path.is_dir()]) if results_root.exists() else []
    latest_run = run_dirs[-1] if run_dirs else None
    strategy_runs.append({
        'strategy': strategy,
        'latest_run': None if latest_run is None else latest_run.name,
        'run_path': None if latest_run is None else str(latest_run),
        'available': latest_run is not None,
    })

print(f'Using byclient root: {byclient_root}')
strategy_runs_df = pd.DataFrame(strategy_runs)
strategy_runs_df

Using byclient root: c:\Users\leono\Desktop\iscte\Tese\FL_MasterThesis\experiments\byclient


,strategy,latest_run,run_path,available
0,zscore,run_20260331_101113,c:\Users\leono\Desktop\iscte\Tese\FL_MasterThe...,True
1,client_zscore,run_20260331_101922,c:\Users\leono\Desktop\iscte\Tese\FL_MasterThe...,True
2,magnitude_features,run_20260331_102905,c:\Users\leono\Desktop\iscte\Tese\FL_MasterThe...,True
3,magnitude_only,run_20260331_104143,c:\Users\leono\Desktop\iscte\Tese\FL_MasterThe...,True
4,robust_clip,run_20260331_105107,c:\Users\leono\Desktop\iscte\Tese\FL_MasterThe...,True


## Load Latest Results

This cell loads the latest client-disjoint run available for each preprocessing strategy. Strategies without results yet are skipped automatically.

In [37]:
global_frames = []
dataset_frames = []

for row in strategy_runs:
    if not row['available']:
        continue

    run_path = Path(row['run_path'])
    metrics_global = pd.read_csv(run_path / 'metrics_global.csv')
    metrics_by_dataset = pd.read_csv(run_path / 'metrics_by_dataset.csv')

    metrics_global['preprocessing_strategy'] = row['strategy']
    metrics_by_dataset['preprocessing_strategy'] = row['strategy']

    metrics_global['model_candidate'] = metrics_global['selected_candidate']
    metrics_by_dataset['model_candidate'] = metrics_by_dataset['selected_candidate']

    metrics_global['selected_candidate_label'] = (
        metrics_global['preprocessing_strategy'] + ' + ' + metrics_global['model'] + ' (' + metrics_global['selected_candidate'] + ')'
    )
    metrics_by_dataset['selected_candidate_label'] = (
        metrics_by_dataset['preprocessing_strategy'] + ' + ' + metrics_by_dataset['model'] + ' (' + metrics_by_dataset['selected_candidate'] + ')'
    )

    global_frames.append(metrics_global)
    dataset_frames.append(metrics_by_dataset)

all_global = pd.concat(global_frames, ignore_index=True) if global_frames else pd.DataFrame()
all_by_dataset = pd.concat(dataset_frames, ignore_index=True) if dataset_frames else pd.DataFrame()

print(f"Loaded {len(all_global)} global rows from {len(global_frames)} by-client strategy runs.")
print(f"Loaded {len(all_by_dataset)} dataset rows from {len(dataset_frames)} by-client strategy runs.")

Loaded 20 global rows from 5 by-client strategy runs.
Loaded 60 dataset rows from 5 by-client strategy runs.


## Final Comparison Table

This table ranks every strategy-model combination from best to worst under the client-disjoint split, using the fall-detection priority: higher PR-AUC, lower miss rate, lower FAR, higher balanced accuracy, and higher F1.

In [38]:
if all_global.empty:
    final_comparison = pd.DataFrame(columns=['selected_candidate'])
else:
    final_comparison = (
        all_global[
            [
                'selected_candidate_label',
                'preprocessing_strategy',
                'model',
                'model_candidate',
                'accuracy',
                'balanced_accuracy',
                'specificity',
                'precision',
                'recall',
                'f1',
                'roc_auc',
                'pr_auc',
                'far',
                'miss_rate',
                'tn',
                'fp',
                'fn',
                'tp',
            ]
        ]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(ranking_columns, ascending=ranking_ascending)
        .reset_index(drop=True)
    )

display_columns = [
    'accuracy',
    'balanced_accuracy',
    'specificity',
    'precision',
    'recall',
    'f1',
    'roc_auc',
    'pr_auc',
    'far',
    'miss_rate',
]

if final_comparison.empty:
    display(final_comparison)
else:
    display(final_comparison.assign(**{column: final_comparison[column].round(4) for column in display_columns}))

,selected_candidate,preprocessing_strategy,model,model_candidate,accuracy,balanced_accuracy,specificity,precision,recall,f1,roc_auc,pr_auc,far,miss_rate,tn,fp,fn,tp
0,magnitude_features + neural_network (mlp_wide),magnitude_features,neural_network,mlp_wide,0.9737,0.9746,0.9722,0.9409,0.9770,0.9586,0.9967,0.9937,0.0278,0.0230,17271,493,185,7842
1,magnitude_features + xgboost (xgb_deep),magnitude_features,xgboost,xgb_deep,0.9776,0.9757,0.9806,0.9577,0.9707,0.9642,0.9962,0.9937,0.0194,0.0293,17420,344,235,7792
2,zscore + xgboost (xgb_deep),zscore,xgboost,xgb_deep,0.9587,0.9562,0.9630,0.9205,0.9494,0.9347,0.9906,0.9841,0.0370,0.0506,17106,658,406,7621
3,zscore + neural_network (mlp_wide),zscore,neural_network,mlp_wide,0.9475,0.9529,0.9385,0.8767,0.9674,0.9198,0.9909,0.9833,0.0615,0.0326,16672,1092,262,7765
4,robust_clip + xgboost (xgb_deep),robust_clip,xgboost,xgb_deep,0.9289,0.9212,0.9417,0.8747,0.9007,0.8875,0.9761,0.9586,0.0583,0.0993,16728,1036,797,7230
5,magnitude_features + knn (knn_k5),magnitude_features,knn,knn_k5,0.9465,0.9460,0.9474,0.8903,0.9446,0.9166,0.9752,0.9410,0.0526,0.0554,16830,934,445,7582
6,robust_clip + neural_network (mlp_wide),robust_clip,neural_network,mlp_wide,0.8957,0.8993,0.8896,0.7882,0.9091,0.8443,0.9637,0.9282,0.1104,0.0909,15803,1961,730,7297
7,zscore + knn (knn_k5),zscore,knn,knn_k5,0.9314,0.9300,0.9338,0.8634,0.9262,0.8937,0.9670,0.9224,0.0662,0.0738,16588,1176,592,7435
8,client_zscore + xgboost (xgb_deep),client_zscore,xgboost,xgb_deep,0.8633,0.8154,0.9422,0.8433,0.6887,0.7582,0.9182,0.8700,0.0578,0.3113,16737,1027,2499,5528
9,robust_clip + knn (knn_k21),robust_clip,knn,knn_k21,0.8669,0.8563,0.8845,0.7641,0.8281,0.7948,0.9344,0.8646,0.1155,0.1719,15712,2052,1380,6647


## Results by Dataset

This view compares all strategy-model combinations separately on KFall, SisFall, and UpFall.

In [39]:
if all_by_dataset.empty:
    dataset_comparison = pd.DataFrame(columns=['dataset', 'selected_candidate'])
else:
    dataset_comparison = (
        all_by_dataset[
            [
                'dataset',
                'selected_candidate_label',
                'preprocessing_strategy',
                'model',
                'model_candidate',
                'accuracy',
                'balanced_accuracy',
                'specificity',
                'precision',
                'recall',
                'f1',
                'roc_auc',
                'pr_auc',
                'far',
                'miss_rate',
            ]
        ]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(['dataset', *ranking_columns], ascending=[True, *ranking_ascending])
        .reset_index(drop=True)
    )

if dataset_comparison.empty:
    display(dataset_comparison)
else:
    display(dataset_comparison.assign(**{column: dataset_comparison[column].round(4) for column in display_columns}))

,dataset,selected_candidate,preprocessing_strategy,model,model_candidate,accuracy,balanced_accuracy,specificity,precision,recall,f1,roc_auc,pr_auc,far,miss_rate
0,KFall,magnitude_features + neural_network (mlp_wide),magnitude_features,neural_network,mlp_wide,0.9906,0.9908,0.9887,0.9857,0.9929,0.9893,0.9994,0.9992,0.0113,0.0071
1,KFall,magnitude_features + xgboost (xgb_deep),magnitude_features,xgboost,xgb_deep,0.9887,0.9885,0.9905,0.9878,0.9865,0.9871,0.9993,0.9991,0.0095,0.0135
2,KFall,robust_clip + xgboost (xgb_deep),robust_clip,xgboost,xgb_deep,0.9677,0.9645,0.9912,0.9882,0.9377,0.9623,0.9972,0.9964,0.0088,0.0623
3,KFall,zscore + neural_network (mlp_wide),zscore,neural_network,mlp_wide,0.9724,0.9736,0.9638,0.9551,0.9833,0.9690,0.9969,0.9960,0.0362,0.0167
4,KFall,zscore + xgboost (xgb_deep),zscore,xgboost,xgb_deep,0.9698,0.9688,0.9771,0.9705,0.9605,0.9655,0.9962,0.9952,0.0229,0.0395
5,KFall,robust_clip + neural_network (mlp_wide),robust_clip,neural_network,mlp_wide,0.9590,0.9564,0.9774,0.9700,0.9355,0.9524,0.9946,0.9927,0.0226,0.0645
6,KFall,magnitude_features + knn (knn_k5),magnitude_features,knn,knn_k5,0.9741,0.9748,0.9691,0.9613,0.9804,0.9707,0.9919,0.9850,0.0309,0.0196
7,KFall,zscore + knn (knn_k5),zscore,knn,knn_k5,0.9570,0.9573,0.9550,0.9435,0.9595,0.9514,0.9850,0.9737,0.0450,0.0405
8,KFall,robust_clip + knn (knn_k21),robust_clip,knn,knn_k21,0.8997,0.8911,0.9611,0.9428,0.8211,0.8778,0.9764,0.9659,0.0389,0.1789
9,KFall,client_zscore + xgboost (xgb_deep),client_zscore,xgboost,xgb_deep,0.8482,0.8354,0.9402,0.9053,0.7306,0.8086,0.9387,0.9314,0.0598,0.2694


## Interpretation of Results

This section summarizes the main conclusions from the client-disjoint strategy-model comparison using the fall-detection ranking.

In [40]:
from IPython.display import Markdown, display

if final_comparison.empty or dataset_comparison.empty:
    display(Markdown('No comparison results are available yet.'))
else:
    best_global = final_comparison.iloc[0]
    runner_up = final_comparison.iloc[1] if len(final_comparison) > 1 else None

    best_per_dataset = dataset_comparison.groupby('dataset', as_index=False).first()
    easiest_dataset = best_per_dataset.sort_values(['pr_auc', 'miss_rate'], ascending=[False, True]).iloc[0]
    hardest_dataset = best_per_dataset.sort_values(['pr_auc', 'miss_rate'], ascending=[True, False]).iloc[0]

    strategy_summary = (
        final_comparison.groupby('preprocessing_strategy', as_index=False)
        .agg(
            mean_pr_auc=('pr_auc', 'mean'),
            mean_miss_rate=('miss_rate', 'mean'),
            mean_far=('far', 'mean'),
        )
        .sort_values(['mean_pr_auc', 'mean_miss_rate', 'mean_far'], ascending=[False, True, True])
        .reset_index(drop=True)
    )

    best_strategy = strategy_summary.iloc[0]
    lowest_far_row = final_comparison.sort_values('far', ascending=True).iloc[0]
    lowest_miss_rate_row = final_comparison.sort_values('miss_rate', ascending=True).iloc[0]
    runner_up_text = (
        '- **Closest competitor:** only one candidate is available.'
        if runner_up is None else
        f"- **Closest competitor:** `{runner_up['selected_candidate']}` is the next best option with PR-AUC = **{runner_up['pr_auc']:.4f}** and miss rate = **{runner_up['miss_rate']:.4f}** under the same unseen-client evaluation."
    )

    summary_lines = [
        '### Key Takeaways',
        '',
        (
            f"- **Best overall combination under the client-disjoint split:** `{best_global['selected_candidate']}` ranks first with PR-AUC = **{best_global['pr_auc']:.4f}**, miss rate = **{best_global['miss_rate']:.4f}**, and FAR = **{best_global['far']:.4f}**. This is the strongest balance between detecting falls on unseen clients and avoiding unnecessary false alarms."
        ),
        runner_up_text,
        (
            f"- **Best preprocessing strategy on average:** `{best_strategy['preprocessing_strategy']}` has the strongest mean performance across its models under the client-disjoint split, with average PR-AUC = **{best_strategy['mean_pr_auc']:.4f}**, average miss rate = **{best_strategy['mean_miss_rate']:.4f}**, and average FAR = **{best_strategy['mean_far']:.4f}**."
        ),
        (
            f"- **Dataset difficulty on unseen clients:** `{easiest_dataset['dataset']}` appears easiest in the current setup (top PR-AUC = **{easiest_dataset['pr_auc']:.4f}**), while `{hardest_dataset['dataset']}` is the most challenging (top PR-AUC = **{hardest_dataset['pr_auc']:.4f}**, miss rate = **{hardest_dataset['miss_rate']:.4f}**)."
        ),
        (
            f"- **Error trade-offs:** `{lowest_miss_rate_row['selected_candidate']}` achieves the lowest miss rate (**{lowest_miss_rate_row['miss_rate']:.4f}**), which is especially important when missing a fall is costly. `{lowest_far_row['selected_candidate']}` achieves the lowest FAR (**{lowest_far_row['far']:.4f}**), which is preferable when minimizing false positives is the priority."
        ),
    ]

    display(Markdown('\n'.join(summary_lines)))

### Key Takeaways

- **Best overall combination under the client-disjoint split:** `magnitude_features + neural_network (mlp_wide)` ranks first with PR-AUC = **0.9937**, miss rate = **0.0230**, and FAR = **0.0278**. This is the strongest balance between detecting falls on unseen clients and avoiding unnecessary false alarms.
- **Closest competitor:** `magnitude_features + xgboost (xgb_deep)` is the next best option with PR-AUC = **0.9937** and miss rate = **0.0293** under the same unseen-client evaluation.
- **Best preprocessing strategy on average:** `magnitude_features` has the strongest mean performance across its models under the client-disjoint split, with average PR-AUC = **0.9232**, average miss rate = **0.0425**, and average FAR = **0.0779**.
- **Dataset difficulty on unseen clients:** `KFall` appears easiest in the current setup (top PR-AUC = **0.9992**), while `UpFall` is the most challenging (top PR-AUC = **0.8961**, miss rate = **0.1723**).
- **Error trade-offs:** `magnitude_features + neural_network (mlp_wide)` achieves the lowest miss rate (**0.0230**), which is especially important when missing a fall is costly. `magnitude_features + xgboost (xgb_deep)` achieves the lowest FAR (**0.0194**), which is preferable when minimizing false positives is the priority.